# Exercise 3 — Multi-Agent RAG: Research + Verifier (Google Colab)

### The story stays the same
Same use case as Exercises 1 and 2: **answer questions about your document**, grounded in
your **`story-llama`** Pinecone index. What changes is *how* we get a trustworthy answer.

### The Day-1 arc, now complete
| Exercise | Who is in control | Retrieval |
|---|---|---|
| 1 — RAG chain | Fixed pipeline | **Always** retrieves |
| 2 — Tool agent | One agent decides | Retrieves **if the agent calls the tool** |
| 3 — **Multi-agent (this one)** | A **Supervisor** routes agents | A **research worker** retrieves; a **verifier** checks it |

### What you'll build
A small research team for defense-grade answers — separation of duties:

```
Start → Supervisor → Condition
                        ├─ research_agent  → drafts an answer using document_search
                        ├─ verifier_agent  → checks the answer is grounded in the sources
                        └─ FINISH → return the verified answer (end)

  verifier says NEEDS_REVISION ↺ Supervisor sends it back to research_agent
```

**The sync point:** `document_search` is the *exact same tool* from Exercise 2, which wraps
the *exact same retriever* from Exercise 1. You are not rebuilding retrieval — you're
coordinating two agents around it.

### Prerequisites
- Ran **Exercise 1** (`story-llama` populated)
- Colab secrets (🔑): `PINECONE_API_KEY` + **either** `GOOGLE_API_KEY` or `OPENAI_API_KEY`
- Set `LLM_PROVIDER` in Step 1 (`"google"` or `"openai"`)


## Step 0 — Install
Same stack as Exercise 2 (LangChain + Pinecone + Gemini).


In [1]:
%pip install -q pinecone langchain langchain-core langchain-google-genai google-generativeai langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 12.0 MB/s eta 0:00:00


## Step 1 — Load API keys & pick LLM provider
Add secrets in 🔑, then set `LLM_PROVIDER` to match the key you have.


In [2]:
LLM_PROVIDER = "google"  # "google" | "openai"

from google.colab import userdata
import os

os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

if LLM_PROVIDER == "google":
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
elif LLM_PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    raise ValueError('LLM_PROVIDER must be "google" or "openai"')

print(f"Keys loaded · LLM provider: {LLM_PROVIDER}")

Keys loaded · LLM provider: google


## Step 2 — Setup
Pinecone index, `document_search` tool, LLM helpers, and shared state (`flow_state`, `conversation_history`, `user_question`).


In [20]:
from pinecone import Pinecone
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
import json
from pydantic import BaseModel, Field

INDEX_NAME, NAMESPACE = "story-llama", "default"
TEMP = 0.3
index = Pinecone(api_key=os.environ["PINECONE_API_KEY"]).Index(INDEX_NAME)

def get_llm(temperature=TEMP, output_parser_schema=None):
    if LLM_PROVIDER == "google":
        from langchain_google_genai import ChatGoogleGenerativeAI
        llm = ChatGoogleGenerativeAI(model="gemma-4-31b-it", temperature=temperature)
    else: # LLM_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=temperature)

    if output_parser_schema:
        return llm.with_structured_output(output_parser_schema)
    return llm

def pinecone_search(query: str, top_k: int = 4) -> list[str]:
    hits = index.search(namespace=NAMESPACE, query={"inputs": {"text": query}, "top_k": top_k}, fields=["chunk_text"])["result"]["hits"]
    return [h["fields"]["chunk_text"] for h in hits]

@tool
def document_search(query: str) -> str:
    """Search and retrieve relevant information from the uploaded document."""
    chunks = pinecone_search(query)
    return "\n\n".join(f"Source {i+1}:\n{t}" for i, t in enumerate(chunks))

def to_text(content) -> str:
    if isinstance(content, str): return content
    if isinstance(content, dict): return content.get("text", str(content))
    if isinstance(content, list): return "".join(to_text(p) for p in content)
    return str(content)

def ask_llm(llm, messages) -> str:
    # This function is now only used for unstructured LLM calls
    return to_text(llm.invoke(messages).content)

# Define the Pydantic model for Supervisor's structured output
class SupervisorOutput(BaseModel):
    next: str = Field(description="The next worker to assign: 'research_agent', 'verifier_agent', or 'FINISH'")
    instructions: str = Field(description="Instructions for the next worker")

flow_state = {"next": "", "instructions": ""}
conversation_history, user_question = [], ""

## Step 3 — Agent nodes
Supervisor, condition router, workers, and FINISH — each in its own cell below.


## Step 4 — Supervisor node
The Supervisor is an LLM that returns **only JSON** deciding who goes next: research first, then always verify, and only FINISH once the verifier approves.


In [21]:
SUPERVISOR_SYSTEM = """You are a supervisor coordinating a research team that answers a
user's question about a document.

Workers:
- research_agent: searches the document and drafts a grounded answer
- verifier_agent: checks whether the draft answer is fully supported by the document

Rules:
1. First assign research_agent to draft an answer.
2. After research_agent, ALWAYS assign verifier_agent.
3. If the verifier's latest message starts with APPROVED, respond with FINISH.
4. If the verifier's latest message starts with NEEDS_REVISION, assign research_agent again
   and put the verifier's feedback in the instructions.
5. Never skip the verifier. Never FINISH before a verification.

Return ONLY valid JSON with exactly these two keys:
{"next": "research_agent" | "verifier_agent" | "FINISH", "instructions": "what the next worker should do"}
No extra text. No markdown. Just JSON."""

def run_supervisor():
    structured_llm = get_llm(output_parser_schema=SupervisorOutput)
    output: SupervisorOutput = structured_llm.invoke([SystemMessage(content=SUPERVISOR_SYSTEM)] + conversation_history)

    flow_state["next"], flow_state["instructions"] = output.next, output.instructions
    # Store the JSON string representation in conversation history
    conversation_history.append(AIMessage(content=output.model_dump_json()))
    print(f"\n[Supervisor] next -> {output.next}")
    print(f"[Supervisor] instructions -> {output.instructions[:120]}...")
    return {"next": output.next, "instructions": output.instructions} # Return dict for consistency with original flow

## Step 5 — Condition node
Pure `if/else` — reads `flow_state['next']` and picks the branch. No AI here.


In [22]:
def check_condition():
    nxt = flow_state["next"]
    if nxt == "research_agent": return "research_agent"
    if nxt == "verifier_agent": return "verifier_agent"
    return "FINISH"

## Step 6 — research_agent
Calls `document_search` on the original question, then drafts a grounded answer. Revises if the Supervisor passed verifier feedback in `instructions`.


In [23]:
RESEARCH_SYSTEM = """You are a research assistant. Answer the user's question using ONLY the
provided document sources. If the sources do not contain the answer, say so plainly.
Quote or reference the relevant sources. Do not invent facts."""

def run_research_agent():
    sources = document_search.invoke(user_question)
    task = (
        f"Question: {user_question}\n\n"
        f"Supervisor instructions:\n{flow_state['instructions']}\n\n"
        f"Document sources:\n{sources}"
    )
    answer = ask_llm(get_llm(), [SystemMessage(content=RESEARCH_SYSTEM), HumanMessage(content=task)])
    conversation_history.append(HumanMessage(content=f"[research_agent draft answer]\n{answer}"))
    print("\n[research_agent]", answer[:400], "...")
    return answer

## Step 7 — verifier_agent
Re-pulls sources and compares them to the latest draft. Must start with **`APPROVED`** or **`NEEDS_REVISION`** so the Supervisor can route on it.


In [24]:
VERIFIER_SYSTEM = """You are a groundedness verifier. You are given the document sources and a
draft answer. Decide whether EVERY claim in the draft is supported by the sources.

Reply format (STRICT):
- Start with 'APPROVED' if the answer is fully grounded, or 'NEEDS_REVISION' if not.
- Then one or two sentences of justification. If NEEDS_REVISION, say exactly what is unsupported."""

def run_verifier_agent():
    sources = document_search.invoke(user_question)
    draft = next(
        (m.content for m in reversed(conversation_history)
         if isinstance(m, HumanMessage) and m.content.startswith("[research_agent draft answer]")),
        ""
    )
    task = f"Question: {user_question}\n\nDocument sources:\n{sources}\n\nDraft answer:\n{draft}"
    verdict = ask_llm(get_llm(temperature=0.0), [SystemMessage(content=VERIFIER_SYSTEM), HumanMessage(content=task)])
    conversation_history.append(HumanMessage(content=f"[verifier_agent]\n{verdict}"))
    print("\n[verifier_agent]", verdict[:400])
    return verdict

## Step 8 — FINISH node
Returns the final verified answer once the Supervisor routes here.


In [25]:
FINISH_SYSTEM = """You are the presenter. The team has produced a verified, grounded answer.
Read the conversation and return the FINAL answer to the user's question in a clear, concise
form. Do not add unverified information."""

def run_finish():
    task = f"User's question: {user_question}\n\nReturn the final verified answer."
    final = ask_llm(get_llm(), [SystemMessage(content=FINISH_SYSTEM)] + conversation_history + [HumanMessage(content=task)])
    print("\n" + "=" * 60 + "\n[FINISH]\n" + "=" * 60)
    print(final)
    return final

## Step 9 — Orchestrator
Full loop with `max_loops` capping research↔verify cycles.


In [26]:
def run_flow(question: str, max_loops: int = 3):
    global conversation_history, flow_state, user_question
    conversation_history, flow_state, user_question = [], {"next": "", "instructions": ""}, question
    print("=" * 60 + f"\nQUESTION: {question}\n" + "=" * 60)
    conversation_history.append(HumanMessage(content=question))
    loop_count = {"research_agent": 0, "verifier_agent": 0}

    while True:
        run_supervisor()
        route = check_condition()
        print(f"\n[Condition] -> {route}")

        if route == "research_agent":
            if loop_count["research_agent"] >= max_loops:
                flow_state["next"] = "FINISH"; continue
            run_research_agent()
            loop_count["research_agent"] += 1
        elif route == "verifier_agent":
            if loop_count["verifier_agent"] >= max_loops:
                flow_state["next"] = "FINISH"; continue
            run_verifier_agent()
            loop_count["verifier_agent"] += 1
        elif route == "FINISH":
            result = run_finish()
            print("\n✅ FLOW COMPLETE")
            return result

## Step 10 — Run the team
Watch: Supervisor → research_agent → verifier_agent → FINISH.


In [27]:
result = run_flow("What is the story about and who are the main characters?", max_loops=3)

QUESTION: What is the story about and who are the main characters?

[Supervisor] next -> research_agent
[Supervisor] instructions -> Search the document to identify the story's plot summary and the main characters....

[Condition] -> research_agent

[research_agent] {'type': 'thinking', 'thinking': '*   Question: What is the story about and who are the main characters?\n    *   Constraint: Use ONLY provided document sources.\n    *   Sources:\n        *   Source 1: Excerpt from Little Red Riding Hood (Girl, mother, Grandma, Wolf). Plot: Girl goes to sick Grandma\'s house with milk, bread, and butter; meets a Wolf in the forest.\n        *   Source 2: Excerpt  ...

[Supervisor] next -> verifier_agent
[Supervisor] instructions -> Verify that the draft answer is fully supported by the provided document sources and that it accurately identifies the p...

[Condition] -> verifier_agent

[verifier_agent] {'type': 'thinking', 'thinking': 'The user wants me to verify if the draft answer is grou

## Step 11 — 🔧 Your turn
1. Ask something only partly in the document — watch `NEEDS_REVISION` loop back to research.
2. Ask something not in the doc — research should say so; verifier should approve an honest "not found".
3. Add a third worker (e.g. `summarizer_agent`) to the Supervisor rules and `check_condition`.


In [ ]:
# result = run_flow("What is the moral of the story?")
# result = run_flow("What is the capital of France?")  # not in the document

result = run_flow("What happens at the end of the story?", max_loops=3)

## ✅ You built a multi-agent RAG system — the arc is complete
Same document, same **`story-llama`** index, same `document_search` tool — three levels of control:

| # | Pattern | Control | Grounding guarantee |
|---|---|---|---|
| 1 RAG | Fixed pipeline | none | retrieved context in prompt |
| 2 Tool agent | agent decides to call the tool | the agent | agent chooses to search |
| 3 Multi-agent | Supervisor routes research + verify | the Supervisor | a **separate verifier** checks it |

### Flowise → code mapping
| Flowise Node | Code |
|---|---|
| Start | reset state, set `user_question` |
| Supervisor | `run_supervisor()` (JSON → Flow State) |
| Condition | `check_condition()` |
| research_agent | `run_research_agent()` → calls `document_search` |
| verifier_agent | `run_verifier_agent()` → APPROVED / NEEDS_REVISION |
| Loop cap | `loop_count` + `max_loops` |
| FINISH | `run_finish()` (terminal) |

### Bridge to Day 2
The verifier here is an **online, per-request** groundedness check. On Day 2 you'll build the
**offline** version — an eval harness that scores a fixed set of questions in batch. Then you
wrap the Exercise-2 agent in FastAPI, Dockerize it, and deploy to Hugging Face.

> **Bonus notebook:** `Exercise_3_Bonus_SoftwareQA_Team_Colab.ipynb` shows the *same* Supervisor
> pattern applied to a software-engineer + QA-engineer coding team — useful to see the pattern
> generalize beyond RAG.
